Custom CNN Baseline (ResNet-18 fine-tuned in PyTorch)


1. Download dataset v2 from Roboflow

In [ ]:
!pip install -q roboflow
import os
from google.colab import files, userdata
from roboflow import Roboflow

DATA_DIR = "/content/data"
if not os.path.exists(f"{DATA_DIR}/train"):
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    project = rf.workspace("vanshikas-workspace-hizw5").project("fabric-detection-pipeline")
    project.version(2).download("folder", location=DATA_DIR)

# Images with no class are exported to "Unlabeled"; they are not part of the two-class task
!rm -rf /content/data/*/Unlabeled
for s in ["train", "valid", "test"]:
    print(s, {c: len(os.listdir(f"{DATA_DIR}/{s}/{c}")) for c in sorted(os.listdir(f"{DATA_DIR}/{s}"))})

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data in folder:: 100%|██████████| 4161/4161 [00:09<00:00, 442.79it/s]


train {'defect free': 1622, 'defective': 1148}
valid {'defect free': 495, 'defective': 292}
test {'defect free': 244, 'defective': 149}


2. Setup, augmentations and data loaders

In [ ]:
import time, random, numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

torch.manual_seed(42); random.seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class Rotate90:
    """Random 90-degree rotation: none, counter-clockwise or clockwise (matches Roboflow's 90° Rotate)."""
    def __call__(self, img):
        k = random.choice([0, 1, 3])
        return img.rotate(90 * k) if k else img

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    Rotate90(),
    transforms.ColorJitter(brightness=0.15),   # matches Roboflow Brightness ±15%
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(f"{DATA_DIR}/train", train_tf)
val_ds   = datasets.ImageFolder(f"{DATA_DIR}/valid", eval_tf)
test_ds  = datasets.ImageFolder(f"{DATA_DIR}/test",  eval_tf)
classes = train_ds.classes
print("Classes:", classes)
print("Train / valid / test:", len(train_ds), len(val_ds), len(test_ds))

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

Using: cuda Tesla T4
Classes: ['defect free', 'defective']
Train / valid / test: 2770 787 393


3. Model: ResNet-18 pretrained on ImageNet, final layer replaced with 2 outputs

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]


Trainable parameters: 11,177,538


4. Train (best epoch kept by validation accuracy)

In [ ]:
def evaluate(loader):
    model.eval(); loss_sum, correct, n = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += criterion(out, y).item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item(); n += x.size(0)
    return loss_sum / n, correct / n

EPOCHS = 10
history, best_acc = [], 0.0
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train(); loss_sum, correct, n = 0, 0, 0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item(); n += x.size(0)
    val_loss, val_acc = evaluate(val_dl)
    history.append([epoch, loss_sum / n, correct / n, val_loss, val_acc])
    print(f"Epoch {epoch:2d} | train loss {loss_sum/n:.4f} acc {correct/n:.4f} | val loss {val_loss:.4f} acc {val_acc:.4f}")
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_resnet18.pt")
train_time = time.time() - start
print(f"\nTraining time: {train_time/60:.1f} min | best val acc: {best_acc:.4f}")

5. Training curves

In [ ]:
h = pd.DataFrame(history, columns=["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])
h.to_csv("training_history.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h.plot(x="epoch", y=["train_loss", "val_loss"], ax=ax[0], title="Loss", marker="o")
h.plot(x="epoch", y=["train_acc", "val_acc"], ax=ax[1], title="Accuracy", marker="o")
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

6. Test-set evaluation

In [ ]:
model.load_state_dict(torch.load("best_resnet18.pt")); model.eval()
y_true, y_pred, y_conf = [], [], []
with torch.no_grad():
    for x, y in test_dl:
        probs = torch.softmax(model(x.to(device)), 1).cpu()
        conf, pred = probs.max(1)
        y_true += y.tolist(); y_pred += pred.tolist(); y_conf += conf.tolist()

test_acc = accuracy_score(y_true, y_pred)
print(f"Test accuracy: {test_acc:.4f}  ({sum(a == b for a, b in zip(y_true, y_pred))}/{len(y_true)})\n")
print(classification_report(y_true, y_pred, target_names=classes, digits=4))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=classes).plot(cmap="Blues")
plt.title("ResNet-18 (PyTorch) - test set"); plt.savefig("confusion_matrix.png", dpi=150); plt.show()

7. Inference latency (one image at a time, after warm-up)

In [ ]:
x0 = test_ds[0][0].unsqueeze(0).to(device)
with torch.no_grad():
    for _ in range(10): model(x0)
times = []
with torch.no_grad():
    for i in range(50):
        x = test_ds[i][0].unsqueeze(0).to(device)
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.perf_counter(); model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        times.append((time.perf_counter() - t) * 1000)
latency = np.mean(times)
print(f"Average latency: {latency:.2f} ms/image on {device}")

8. Predictions on 50 held-out test images

Same selection as `infer.py` (seed 42, ImageFolder order), and the list is saved to `test_50.txt`,
so both models are scored on identical images.

In [ ]:
idx50 = random.Random(42).sample(range(len(test_ds)), 50)
rows = []
for i in idx50:
    path = test_ds.samples[i][0]
    rows.append({"image_name": os.path.basename(path), "actual_label": classes[y_true[i]],
                 "predicted_label": classes[y_pred[i]], "confidence": round(y_conf[i], 4),
                 "correct": y_true[i] == y_pred[i]})
df = pd.DataFrame(rows)
df.to_csv("cnn_predictions.csv", index=False)
with open("test_50.txt", "w") as f:
    f.write("\n".join(os.path.relpath(test_ds.samples[i][0], DATA_DIR) for i in idx50))
print(f"50-image accuracy: {df.correct.mean():.2%}")
df

9. Misclassified test images (failure analysis)

In [ ]:
wrong = [i for i in range(len(y_true)) if y_true[i] != y_pred[i]]
print(f"Misclassified: {len(wrong)} of {len(y_true)}")
show = sorted(wrong, key=lambda i: -y_conf[i])[:8]   # most confident mistakes first
if show:
    fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 3.4), squeeze=False)
    for ax, i in zip(axes[0], show):
        ax.imshow(plt.imread(test_ds.samples[i][0])); ax.axis("off")
        ax.set_title(f"true: {classes[y_true[i]]}\npred: {classes[y_pred[i]]} ({y_conf[i]:.2f})", fontsize=9)
    plt.tight_layout(); plt.savefig("misclassified.png", dpi=150); plt.show()
pd.DataFrame([{"image_name": os.path.basename(test_ds.samples[i][0]), "actual": classes[y_true[i]],
               "predicted": classes[y_pred[i]], "confidence": round(y_conf[i], 4)} for i in wrong])

10. Summary for the report

*   List item
*   List item



In [ ]:
summary = pd.DataFrame([
    ["Test accuracy", f"{test_acc:.2%}"],
    ["50-image accuracy", f"{df.correct.mean():.2%}"],
    ["Best validation accuracy", f"{best_acc:.2%}"],
    ["Training time", f"{train_time/60:.1f} min ({EPOCHS} epochs, {device})"],
    ["Inference latency", f"{latency:.2f} ms/image (local {device})"],
], columns=["Metric", "ResNet-18 (PyTorch)"])
summary.to_csv("cnn_summary.csv", index=False)
summary

11. Download all outputs

In [ ]:
!zip -q cnn_results.zip best_resnet18.pt training_history.csv training_curves.png confusion_matrix.png cnn_predictions.csv test_50.txt misclassified.png cnn_summary.csv
files.download("cnn_results.zip")